# Mutatieconverter amateurvoetbal

Dit notebook leest een geüpload Excel-bestand met mutaties in en maakt een `txt`-uitvoerbestand volgens de afgesproken opmaak.

Belangrijk:
- kolom **G** = vereniging
- kolom **H** = divisie of klasse
- kolom **L** = trainer
- kolommen **R t/m AD** = nieuwe spelers
- kolommen **AE t/m AQ** = vertrokken spelers

Ingebouwde normalisaties:
- `speler - club` wordt `speler, club`
- buitenlandse clubs met land tussen haakjes worden omgezet om dubbele haakjes te voorkomen
- als de club al tussen haakjes staat, komt er geen extra paar haakjes omheen
- punten aan het einde van namen of clubs worden weggehaald


In [ ]:
from google.colab import files
uploaded = files.upload()
uploaded_filename = next(iter(uploaded))
print(f"Bestand geüpload: {uploaded_filename}")

In [ ]:
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, OrderedDict
from pathlib import Path


NS = {
    "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
}


def clean_whitespace(text: str) -> str:
    text = str(text or "")
    text = text.replace("\r", "\n").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def strip_trailing_periods(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"\.(?=\s*(?:,|\)|$))", "", text)
    return text.strip()


def normalize_country_parens(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"\s*\(([^()]+)\)\s*$", lambda m: ", " + m.group(1).strip(), text)
    text = re.sub(r"\s*,\s*", ", ", text)
    return text.strip(" ,")


def normalize_existing_parenthetical_entry(entry: str) -> str:
    entry = clean_whitespace(entry)
    match = re.match(r"^(.*?)\s*\((.*)\)\s*$", entry)
    if not match:
        return entry

    name = strip_trailing_periods(clean_whitespace(match.group(1)))
    club = strip_trailing_periods(clean_whitespace(match.group(2)))
    club = normalize_country_parens(club)
    club = strip_trailing_periods(club)
    return f"{name} ({club})"


def normalize_plain_entry(entry: str) -> list[str]:
    entry = clean_whitespace(entry)
    if not entry:
        return []

    entry = re.sub(r"\s+-\s+", ", ", entry)
    entry = re.sub(r"\s*,\s*", ", ", entry)
    entry = strip_trailing_periods(entry)

    if "," not in entry:
        return [normalize_existing_parenthetical_entry(entry)]

    tokens = [strip_trailing_periods(token.strip()) for token in entry.split(",") if token.strip()]
    if not tokens:
        return []

    if len(tokens) >= 3 and tokens[-1].lower().startswith("allen ") and all(len(token.split()) >= 2 for token in tokens[:-1]):
        club = normalize_country_parens(tokens[-1].strip("() "))
        return [f"{strip_trailing_periods(name)} ({club})" for name in tokens[:-1]]

    if len(tokens) == 2:
        name = tokens[0]
        club = tokens[1]
    else:
        if len(tokens[0].split()) >= 2:
            name = tokens[0]
            club = ", ".join(tokens[1:])
        else:
            name = " ".join(tokens[:-1])
            club = tokens[-1]

    name = strip_trailing_periods(clean_whitespace(name))
    club = clean_whitespace(club)

    if re.fullmatch(r"\(.*\)", club):
        club = club[1:-1].strip()

    club = strip_trailing_periods(club)
    club = normalize_country_parens(club)
    club = strip_trailing_periods(club)

    return [f"{name} ({club})"]


def normalize_cell_value(cell_value: str) -> list[str]:
    if cell_value is None:
        return []

    text = clean_whitespace(cell_value)
    if not text:
        return []

    items = []
    for part in text.split("\n"):
        part = clean_whitespace(part)
        if not part:
            continue
        items.extend(normalize_plain_entry(part))
    return [item for item in items if item]


def join_player_fields(values: list[str]) -> str:
    items = []
    for value in values:
        items.extend(normalize_cell_value(value))
    return ", ".join(items) if items else "niemand"


def col_range(start: str, end: str) -> list[str]:
    def to_num(col: str) -> int:
        number = 0
        for char in col:
            number = number * 26 + ord(char) - 64
        return number

    def to_col(number: int) -> str:
        result = ""
        while number:
            number, remainder = divmod(number - 1, 26)
            result = chr(65 + remainder) + result
        return result

    return [to_col(i) for i in range(to_num(start), to_num(end) + 1)]


def load_first_sheet_rows(xlsx_path: str | Path) -> dict[int, dict[str, str]]:
    xlsx_path = Path(xlsx_path)
    workbook = zipfile.ZipFile(xlsx_path)

    shared_strings = []
    if "xl/sharedStrings.xml" in workbook.namelist():
        shared_root = ET.fromstring(workbook.read("xl/sharedStrings.xml"))
        for item in shared_root.findall("a:si", NS):
            parts = [node.text or "" for node in item.iterfind(".//a:t", NS)]
            shared_strings.append("".join(parts))

    sheet_root = ET.fromstring(workbook.read("xl/worksheets/sheet1.xml"))
    rows = defaultdict(dict)

    for cell in sheet_root.findall(".//a:sheetData/a:row/a:c", NS):
        reference = cell.attrib["r"]
        match = re.match(r"([A-Z]+)(\d+)", reference)
        column = match.group(1)
        row_number = int(match.group(2))

        cell_type = cell.attrib.get("t")
        value_node = cell.find("a:v", NS)
        inline_node = cell.find("a:is", NS)

        if cell_type == "s" and value_node is not None:
            value = shared_strings[int(value_node.text)]
        elif cell_type == "inlineStr" and inline_node is not None:
            value = "".join(node.text or "" for node in inline_node.iterfind(".//a:t", NS))
        elif value_node is not None:
            value = value_node.text
        else:
            value = ""

        rows[row_number][column] = value

    return rows


RANK_ORDER = {
    "eerste": 1,
    "tweede": 2,
    "derde": 3,
    "vierde": 4,
    "vijfde": 5,
}


def normalize_class_for_matching(label: str) -> str:
    label = clean_whitespace(label).lower()
    label = label.replace("klassse", "klasse")
    return label


def class_sort_key(label: str) -> tuple:
    normalized = normalize_class_for_matching(label)

    if normalized == "derde divisie b":
        return (0, 0, "")
    if normalized == "vierde divisie c":
        return (1, 0, "")
    if normalized == "vrouwen hoofdklasse":
        return (3, 0, "")
    if normalized == "vrouwen eerste klasse c":
        return (3, 1, "")
    if normalized == "vrouwen eerste klasse d":
        return (3, 2, "")

    match = re.match(r"^(eerste|tweede|derde|vierde|vijfde)\s+klasse\s+([a-z])$", normalized)
    if match:
        return (2, RANK_ORDER[match.group(1)], match.group(2))

    return (2, 99, normalized)


def excel_to_txt_mutaties(xlsx_path: str | Path) -> str:
    rows = load_first_sheet_rows(xlsx_path)
    items = []

    for row_number in sorted(rows):
        if row_number == 1:
            continue

        row = rows[row_number]
        club = clean_whitespace(row.get("G", ""))
        division = clean_whitespace(row.get("H", ""))
        if not club or not division:
            continue

        trainer = strip_trailing_periods(clean_whitespace(row.get("L", "")))
        nieuwe_spelers = join_player_fields([row.get(col, "") for col in col_range("R", "AD")])
        vertrokken_spelers = join_player_fields([row.get(col, "") for col in col_range("AE", "AQ")])

        items.append({
            "club": club,
            "division": division,
            "trainer": trainer,
            "nieuwe_spelers": nieuwe_spelers,
            "vertrokken_spelers": vertrokken_spelers,
        })

    groups = defaultdict(list)
    original_labels = OrderedDict()

    for item in items:
        key = normalize_class_for_matching(item["division"])
        groups[key].append(item)
        original_labels.setdefault(key, item["division"])

    ordered_keys = sorted(groups.keys(), key=lambda key: class_sort_key(original_labels[key]))

    lines = ["<body>"]

    for key in ordered_keys:
        label = original_labels[key]
        lines.append(f"<subhead_lead>{label}</subhead_lead>")

        for index, item in enumerate(groups[key]):
            if index > 0:
                lines.append("<EP,1>")

            lines.append(f"<subhead>{item['club']}</subhead>")
            lines.append(f"<howto_facts><bold><CO,5>Nieuw: </bold>{item['nieuwe_spelers']}</howto_facts>")
            lines.append(f"<howto_facts><bold><CO,5>Vertrokken: </bold>{item['vertrokken_spelers']}</howto_facts>")
            lines.append(f"<howto_facts><bold><CO,5>Trainer: </bold>{item['trainer']}</howto_facts>")

    lines.append("</body>")
    return "\n".join(lines)


if __name__ == "__main__":
    input_path = Path("TESTBESTAND DL _ Amateurvoetbalseizoen 2026-2027.xlsx")
    output_path = Path("mutaties_output.txt")
    output_path.write_text(excel_to_txt_mutaties(input_path), encoding="utf-8")
    print(f"Bestand gemaakt: {output_path}")


output_filename = "mutaties_output.txt"
output_text = excel_to_txt_mutaties(uploaded_filename)

with open(output_filename, "w", encoding="utf-8") as handle:
    handle.write(output_text)

print(output_text[:3000])
print("\n--- EINDE VAN VOORBEELD ---")
files.download(output_filename)
